In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
df = pd.read_csv('used_device_data.csv')
df
df.columns
df.describe()
df.dtypes
df.shape
duplicated_rows= df[df.duplicated()]
duplicated_rows.shape
df.isnull().sum()
import seaborn as sns
import matplotlib.pyplot as plt 

features=['rear_camera_mp','front_camera_mp','internal_memory','ram','battery','weight']

for i in features:
    sns.boxplot(df[i])
    plt.show()

for i in features:
    a=df[i].median()
    print(i,' median value : ', a)
    df[i]= df[i].fillna(a)

df.isnull().sum()
    
num_features=[feature for feature in df.columns if df[feature].dtype != 'O']
num_features

cat_features=[feature for feature in df.columns if df[feature].dtype == 'O']
cat_features

fig,axs= plt.subplots(5,3,figsize=(15, 15))
# Flatten the 2D array of subplots to make them easier to access
axs = axs.ravel()
for i, feature in enumerate(df.columns):
    # Plot a histogram of the feature in the current subplot
    axs[i].hist(df[feature])
    axs[i].set_title(feature)


# Adjust the spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()


fig= plt.figure(figsize=(15,10))
sns.countplot(x=df.device_brand,order=df["device_brand"].value_counts().index)
plt.xticks(rotation=90)


fig,axs= plt.subplots(4,3,figsize=(15, 15))
# Flatten the 2D array of subplots to make them easier to access
axs = axs.ravel()
for i, feature in enumerate(num_features):
    
    axs[i].boxplot(df[feature])
    axs[i].set_title(feature)

# method used to remove an axis from a figure; here removing last grid--> 12th grid since we only have 11 numerical features
fig.delaxes(axs[-1])
# Adjust the spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

features=['screen_size',
 'front_camera_mp',
 'ram',
 'battery',
 'weight',
 'normalized_used_price',
 'normalized_new_price']
for i in features:
    lower = df[i].quantile(0.10)
    upper = df[i].quantile(0.90)
    df[i] = np.where(df[i] <lower, lower,df[i])
    df[i] = np.where(df[i] >upper, upper,df[i])
    print('Feature: ',i)
    print('Skewness value: ',df[i].skew())
    print('\n')

fig,axs= plt.subplots(4,3,figsize=(15, 15))
# Flatten the 2D array of subplots to make them easier to access
axs = axs.ravel()
for i, feature in enumerate(num_features):
    
    axs[i].boxplot(df[feature])
    axs[i].set_title(feature)

# method used to remove an axis from a figure; here removing last grid--> 12th grid since we only have 11 numerical features
fig.delaxes(axs[-1])
# Adjust the spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()

upper= df.rear_camera_mp.quantile(0.95)
df.rear_camera_mp= np.where(df.rear_camera_mp>upper,upper,df.rear_camera_mp)

upper= df.internal_memory.quantile(0.9)
df.internal_memory= np.where(df.internal_memory>upper,upper,df.internal_memory)

upper= df.weight.quantile(0.8)
df.weight= np.where(df.weight>upper,upper,df.weight)


fig,axs= plt.subplots(4,3,figsize=(15, 15))
# Flatten the 2D array of subplots to make them easier to access
axs = axs.ravel()
for i, feature in enumerate(num_features):
    
    axs[i].boxplot(df[feature])
    axs[i].set_title(feature)

# method used to remove an axis from a figure; here removing last grid--> 12th grid since we only have 11 numerical features
fig.delaxes(axs[-1])
# Adjust the spacing between subplots
plt.tight_layout()
# Show the plot
plt.show()


bi_num=['screen_size',
 'rear_camera_mp',
 'front_camera_mp',
 'internal_memory',
 'ram',
 'battery',
 'weight',
 'release_year',
 'days_used',
 'normalized_new_price']

fig,axs= plt.subplots(5,2,figsize=(12,12))
axs=axs.ravel()
for i,ax in enumerate(axs):

    sns.regplot(x=bi_num[i],y='normalized_used_price',data=df,ax=ax,color='black',scatter_kws={"color":"blue"})

plt.tight_layout()
plt.show()

X= df.drop('normalized_used_price',axis=1)
y= df.normalized_used_price

# Label encoding for categoricals
for colname in X.select_dtypes("object"):
    X[colname], _ = X[colname].factorize()

# All discrete features should now have integer/float dtypes (double-check this before using MI!)
X.dtypes == object

from sklearn.feature_selection import mutual_info_regression


mi_scores = mutual_info_regression(X, y)
mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
mi_scores = mi_scores.sort_values(ascending=False)

mi_scores  # show a few features with their MI scores

from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split

# dataset split
# creating a function for dataset split
def dataset(X,y):
    train_full_X, val_X, train_full_y, val_y = train_test_split(X, y,test_size=0.2,random_state = 0)

# Use the same function above for the validation set
    train_X, test_X, train_y, test_y = train_test_split(train_full_X, train_full_y, test_size=0.25,random_state = 0)
    return (train_X, val_X, train_y, val_y)

scores=[]

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

X= df.drop('normalized_used_price',axis=1)
y= df.normalized_used_price

X=pd.get_dummies(X)


#X= scale(X)
#X= PCAx(X)

X_train,X_val,y_train,y_val= dataset(X,y)

decision_model1= DecisionTreeRegressor(random_state=1)
decision_model1.fit(X_train, y_train)
preds= decision_model1.predict(X_val)

r2= r2_score(y_val,preds)
MSE= mean_squared_error(y_val,preds)


a= "Decision Tree Regreesion with all features"
print(a)
print("R2: ",round(r2,4))
print("MSE: ",round(MSE,4))

scores.append((a,round(r2,4),round(MSE,4)))


from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# ===============================
# 1. Prepare Data
# ===============================
X = df.drop('normalized_used_price', axis=1)
y = df['normalized_used_price']

X = pd.get_dummies(X)

# ===============================
# 2. Split Data (Make sure dataset() is correct)
# ===============================
X_train, X_val, y_train, y_val = dataset(X, y)

# ===============================
# 3. Train Model (IMPORTANT)
# ===============================
linear_model1 = LinearRegression()
linear_model1.fit(X_train, y_train)

# ===============================
# 4. Predict
# ===============================
preds = linear_model1.predict(X_val)

# ===============================
# 5. Evaluate
# ===============================
r2 = r2_score(y_val, preds)
MSE = mean_squared_error(y_val, preds)

a = "Linear Regression with all features"

print(a)
print("R2:", round(r2, 4))
print("MSE:", round(MSE, 4))

# ===============================
# 6. Store Result (Ensure list exists)
# ===============================

scores.append((a, round(r2, 4), round(MSE, 4)))

scores

import joblib

joblib.dump(linear_model1, "model.pkl")
joblib.dump(X.columns.tolist(), "columns.pkl")